# **Generación de imagenes de explicabilidad**

Este notebook es uno de los principales, aquí tomamos un modelo predictivo y generamos las imagenes explicativas del conjunto test usando todas las técnicas de la librería.

Para cambiar de modelo, basta cambiar las configuraciones inciales indicando el modelo y el dataset.

**Necesario archivo auxiliar Load_model.py y todos los métodos explicativos de la librería** `glassbox_vit`, o en su defecto, tenerla instalada.

In [ ]:
import os
import io
import shutil
import torch
import numpy as np
from PIL import Image as PILImage
# from datasets import load_dataset as HF
from datasets import Image as HFImage
from transformers import AutoImageProcessor, AutoModelForImageClassification
from huggingface_hub import batch_bucket_files

from Load_model import load_model,load_dataset
from glassbox_vit.lime_explainer import LimeExplainer
from glassbox_vit.shap_explainer import ShapExplainer
from glassbox_vit.rise_explainer import RiseExplainer
from glassbox_vit.patch_explainer import PatchOcclusionExplainer
from glassbox_vit.rollout_explainer import AttentionRolloutExplainer
from glassbox_vit.gradcam_explainer import GradCamExplainer
from glassbox_vit.pca_explainer import PCAExplainer
from glassbox_vit.smoothgrad_explainer import SmoothGradExplainer
from glassbox_vit.expected_gradients_explainer import ExpectedGradientsExplainer
from glassbox_vit.integrated_gradients_explainer import IntegratedGradientsExplainer
from glassbox_vit.scorecam_explainer import ScoreCamExplainer

In [ ]:
# ---  CONFIGURACIÓN INICIAL ---
nombre_modelo = "dataset2_facebook_dinov2"
bucket_id = "z72pepee/tfm-explicabilidad"

# Primero cargamos el modelo
print(" Cargando modelo y procesador...")
procesador, modelo = load_model(nombre_modelo,output_attentions=True,output_hidden_states=True)

# Luego lo pasamos a la GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Usando dispositivo: {device}")
modelo.to(device)
modelo.eval()


# ---  CARGAR DATASET ---
print(" Cargando dataset...")
data = load_dataset("fitzpatrick_stratified_ninelabel_scale")

 Cargando modelo y procesador...
 Carpeta detectada: checkpoint-19530
 Cargando procesador y modelo desde: Modelos_Entrenados/experimento_dataset2_facebook_dinov2/checkpoint-19530...


¡Descarga completada con éxito!


Loading weights:   0%|          | 0/225 [00:00<?, ?it/s]

 Carga en memoria completada con éxito
 Usando dispositivo: cuda
 Cargando dataset...


README.md:   0%|          | 0.00/35.4k [00:00<?, ?B/s]

fitzpatrick_stratified_ninelabel_scale/t(…): reconstructing file:   0%|          |  0.00B /  120MB            

... (4 outputs left out)

fitzpatrick_stratified_ninelabel_scale/t(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14865 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/826 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/826 [00:00<?, ? examples/s]

In [ ]:
# ---  FUNCIÓN PUENTE  ---
def funcion_prediccion_hf(images_numpy):
    imgs_pil = [PILImage.fromarray((img).astype('uint8')) for img in images_numpy]
    inputs = procesador(images=imgs_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = modelo(**inputs)
    return torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()


In [ ]:
# Dependencia: lime (instalar con pip)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=5d80188e293efacfab061c0a4ce5b2f9cb43c87c435e039b149410442ea4b5a0
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
# ---  INICIALIZAR LA LIBRERÍA ---
explicador= LimeExplainer(prediction_function=funcion_prediccion_hf)



carpeta_salida = "./Resultados_LIME/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = []

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando LIME para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil, num_samples=500)

        # Guardar en local
        imagen_final = resultado_xai['lime_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_LIME/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")


 Procesando Split: test
[1/115] Generando LIME para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)


  0%|          | 0/500 [00:00<?, ?it/s]

[2/115] Generando LIME para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)


  0%|          | 0/500 [00:00<?, ?it/s]

[3/115] Generando LIME para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)


  0%|          | 0/500 [00:00<?, ?it/s]

... (9 outputs left out)

  ...LIME/PAT_1259_892_793.png: 100%|##########| 10.8kB / 10.8kB            

  ..._LIME/PAT_1017_97_577.png: 100%|##########| 14.0kB / 14.0kB            

  ...IME/PAT_1300_1056_532.png: 100%|##########| 17.7kB / 17.7kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_LIME/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_LIME/", ignore_errors=True)

In [ ]:
# Dependencia: shap (instalar con pip)

In [ ]:

class_names = list(modelo.config.id2label.values())

# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT (ShapExplainer)...")
explicador = ShapExplainer(
    prediction_function=funcion_prediccion_hf,
    class_names=class_names
)




carpeta_salida = "./Resultados_SHAP/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando SHAP para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil, max_evals=500)

        # Guardar en local
        imagen_final = resultado_xai['shap_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_SHAP/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT (ShapExplainer)...

 Procesando Split: test
[1/115] Generando SHAP para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:12, 12.25s/it]               


[2/115] Generando SHAP para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)


  0%|          | 0/498 [00:00<?, ?it/s]

[3/115] Generando SHAP para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)


... (10 outputs left out)

  ...HAP/PAT_1300_1056_532.png: 100%|##########| 50.9kB / 50.9kB            

  ...HAP/PAT_1374_1285_735.png: 100%|##########| 49.8kB / 49.8kB            

  ...SHAP/PAT_1259_892_793.png: 100%|##########| 48.0kB / 48.0kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_SHAP/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_SHAP/", ignore_errors=True)

In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT (RiseExplainer)...")
explicador = RiseExplainer(
    prediction_function=funcion_prediccion_hf)




carpeta_salida = "./Resultados_RISE/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando RISE para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil,num_masks=2000,batch_size=32)

        # Guardar en local
        imagen_final = resultado_xai['rise_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_RISE/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT (RiseExplainer)...

 Procesando Split: test
[1/115] Generando RISE para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)
[2/115] Generando RISE para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)
[3/115] Generando RISE para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)
[4/115] Generando RISE para: PAT_1300_1056_532.png (Etiqueta real: 0_ACK)
[5/115] Generando RISE para: PAT_1374_1285_735.png (Etiqueta real: 0_ACK)

 Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!

 Subiendo 5 imágenes al bucket 'z72pepee/tfm-explicabilidad'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...RISE/PAT_1259_892_793.png: 100%|##########| 62.8kB / 62.8kB            

  ...ISE/PAT_1300_1056_532.png: 100%|##########| 67.9kB / 67.9kB            

  ...ISE/PAT_1374_1285_735.png: 100%|##########| 65.8kB / 65.8kB            

  ...RISE/PAT_1172_622_420.png: 100%|##########| 77.7kB / 77.7kB            

  ..._RISE/PAT_1017_97_577.png: 100%|##########| 65.4kB / 65.4kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_RISE/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_RISE/", ignore_errors=True)

In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT ...")
explicador = PatchOcclusionExplainer(
    prediction_function=funcion_prediccion_hf)




carpeta_salida = "./Resultados_PATCH_OCCLUSION/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando PATCH OCCLUSION para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil,batch_size=32,patch_size=16,mask_value=0)

        # Guardar en local
        imagen_final = resultado_xai['patch_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_PATCH_OCCLUSION/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT ...

 Procesando Split: test
[1/115] Generando PATCH OCCLUSION para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)
[2/115] Generando PATCH OCCLUSION para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)
[3/115] Generando PATCH OCCLUSION para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)
[4/115] Generando PATCH OCCLUSION para: PAT_1300_1056_532.png (Etiqueta real: 0_ACK)
[5/115] Generando PATCH OCCLUSION para: PAT_1374_1285_735.png (Etiqueta real: 0_ACK)

 Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!

 Subiendo 5 imágenes al bucket 'z72pepee/tfm-explicabilidad'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...SION/PAT_1259_892_793.png: 100%|##########| 43.6kB / 43.6kB            

  ...ION/PAT_1300_1056_532.png: 100%|##########| 53.8kB / 53.8kB            

  ...USION/PAT_1017_97_577.png: 100%|##########| 52.4kB / 52.4kB            

  ...SION/PAT_1172_622_420.png: 100%|##########| 68.4kB / 68.4kB            

  ...ION/PAT_1374_1285_735.png: 100%|##########| 53.9kB / 53.9kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_PATCH_OCCLUSION/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_PATCH_OCCLUSION/", ignore_errors=True)

In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT ...")
explicador = AttentionRolloutExplainer(
    model=modelo,
    processor=procesador
)




carpeta_salida = "./Resultados_ATTENTION_ROLLOUT/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando ATTENCION ROLLOUT para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil)

        # Guardar en local
        imagen_final = resultado_xai['rollout_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_ATTENTION_ROLLOUT/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT ...

 Procesando Split: test
[1/115] Generando ATTENCION ROLLOUT para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)
[2/115] Generando ATTENCION ROLLOUT para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)
[3/115] Generando ATTENCION ROLLOUT para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)
[4/115] Generando ATTENCION ROLLOUT para: PAT_1300_1056_532.png (Etiqueta real: 0_ACK)
[5/115] Generando ATTENCION ROLLOUT para: PAT_1374_1285_735.png (Etiqueta real: 0_ACK)

 Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!

 Subiendo 5 imágenes al bucket 'z72pepee/tfm-explicabilidad'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...LLOUT/PAT_1017_97_577.png: 100%|##########| 60.5kB / 60.5kB            

  ...LOUT/PAT_1172_622_420.png: 100%|##########| 72.8kB / 72.8kB            

  ...OUT/PAT_1300_1056_532.png: 100%|##########| 65.9kB / 65.9kB            

  ...OUT/PAT_1374_1285_735.png: 100%|##########| 61.9kB / 61.9kB            

  ...LOUT/PAT_1259_892_793.png: 100%|##########| 60.0kB / 60.0kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_ATTENTION_ROLLOUT/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_ATTENTION_ROLLOUT/", ignore_errors=True)

In [ ]:
# Dependencia: grad-cam (instalar con pip)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 78.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44284 sha256=0d7c2770d068ba080a5bb1e68aab814fdff9674037ada8a8d35a4b6e2602b7fd
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT ...")
explicador = GradCamExplainer(
    model=modelo,
    processor=procesador
)




carpeta_salida = "./Resultados_GRADCAM/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando GRAD CAM para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil)

        # Guardar en local
        imagen_final = resultado_xai['gradcam_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_GRADCAM/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT ...

 Procesando Split: test
[1/115] Generando GRAD CAM para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)
[2/115] Generando GRAD CAM para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)
[3/115] Generando GRAD CAM para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)
[4/115] Generando GRAD CAM para: PAT_1300_1056_532.png (Etiqueta real: 0_ACK)
[5/115] Generando GRAD CAM para: PAT_1374_1285_735.png (Etiqueta real: 0_ACK)

 Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!

 Subiendo 5 imágenes al bucket 'z72pepee/tfm-explicabilidad'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...DCAM/PAT_1259_892_793.png: 100%|##########| 58.6kB / 58.6kB            

  ...CAM/PAT_1300_1056_532.png: 100%|##########| 54.1kB / 54.1kB            

  ...DCAM/PAT_1172_622_420.png: 100%|##########| 81.3kB / 81.3kB            

  ...CAM/PAT_1374_1285_735.png: 100%|##########| 56.2kB / 56.2kB            

  ...ADCAM/PAT_1017_97_577.png: 100%|##########| 62.1kB / 62.1kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_GRADCAM/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_GRADCAM/", ignore_errors=True)

In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT ...")
explicador = PCAExplainer(
    model=modelo,
    processor=procesador
)




carpeta_salida = "./Resultados_PCA/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando PCA para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil)

        # Guardar en local
        imagen_final = resultado_xai['pca_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_PCA/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT ...

 Procesando Split: test
[1/115] Generando PCA para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)
[2/115] Generando PCA para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)
[3/115] Generando PCA para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)
[4/115] Generando PCA para: PAT_1300_1056_532.png (Etiqueta real: 0_ACK)
[5/115] Generando PCA para: PAT_1374_1285_735.png (Etiqueta real: 0_ACK)

 Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!

 Subiendo 5 imágenes al bucket 'z72pepee/tfm-explicabilidad'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...PCA/PAT_1374_1285_735.png: 100%|##########| 1.64kB / 1.64kB            

  ...s_PCA/PAT_1017_97_577.png: 100%|##########| 1.65kB / 1.65kB            

  ...PCA/PAT_1300_1056_532.png: 100%|##########| 1.64kB / 1.64kB            

  ..._PCA/PAT_1172_622_420.png: 100%|##########| 1.66kB / 1.66kB            

  ..._PCA/PAT_1259_892_793.png: 100%|##########| 1.64kB / 1.64kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_PCA/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_PCA/", ignore_errors=True)

In [ ]:
# Dependencia: captum (instalar con pip)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 9.4 MB/s eta 0:00:00


In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT ...")
explicador = SmoothGradExplainer(
    model=modelo,
    processor=procesador
)




carpeta_salida = "./Resultados_SMOOTHGRAD/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando SMOOTH GRAD para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil,num_samples=50, stdevs=0.15)

        # Guardar en local
        imagen_final = resultado_xai['smoothgrad_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_SMOOTHGRAD/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT ...

 Procesando Split: test
[1/826] Generando SMOOTH GRAD para: httpwwwatlasdermatologicocombrimgimageId1422.jpg (Etiqueta real: 000_benign_dermal)
[2/826] Generando SMOOTH GRAD para: httpwwwatlasdermatologicocombrimgimageId1423.jpg (Etiqueta real: 000_benign_dermal)
[3/826] Generando SMOOTH GRAD para: httpwwwatlasdermatologicocombrimgimageId2165.jpg (Etiqueta real: 000_benign_dermal)
[4/826] Generando SMOOTH GRAD para: httpwwwatlasdermatologicocombrimgimageId2169.jpg (Etiqueta real: 000_benign_dermal)
[5/826] Generando SMOOTH GRAD para: httpwwwatlasdermatologicocombrimgimageId2172.jpg (Etiqueta real: 000_benign_dermal)
[6/826] Generando SMOOTH GRAD para: httpwwwatlasdermatologicocombrimgimageId4150.jpg (Etiqueta real: 000_benign_dermal)
[7/826] Generando SMOOTH GRAD para: httpwwwatlasdermatologicocombrimgimageId4158.jpg (Etiqueta real: 000_benign_dermal)
[8/826] Generando SMOOTH GRAD para: httpwwwatlasdermatologicocombrimgimageId4549.jpg (Etiqueta real: 000

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...hotodrugeruption20jpg.jpg:  10%|9         |   642B / 6.49kB            

  ...temicscleroderma28jpg.jpg:  10%|9         |   466B / 4.72kB            

  ...akeratoacanthoma22jpg.jpg:  10%|9         |   634B / 6.41kB            

... (4 outputs left out)

  ...dicadiabeticorum76jpg.jpg:  10%|9         |   794B / 8.03kB            

  ...scutiscarcinoides6jpg.jpg:  10%|9         |   566B / 5.73kB            

  ...redemascleredema71jpg.jpg:  10%|9         |   780B / 7.88kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_SMOOTHGRAD/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_SMOOTHGRAD/", ignore_errors=True)

In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT ...")
explicador = ExpectedGradientsExplainer(
    model=modelo,
    processor=procesador
)




carpeta_salida = "./Resultados_EXPECTED_GRADIENTS/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando EXPECTED GRADIENTS para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil,baselines=None, n_samples=50)

        # Guardar en local
        imagen_final = resultado_xai['eg_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_EXPECTED_GRADIENTS/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT ...

 Procesando Split: test
[1/115] Generando EXPECTED GRADIENTS para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)
[2/115] Generando EXPECTED GRADIENTS para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)
[3/115] Generando EXPECTED GRADIENTS para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)
[4/115] Generando EXPECTED GRADIENTS para: PAT_1300_1056_532.png (Etiqueta real: 0_ACK)
[5/115] Generando EXPECTED GRADIENTS para: PAT_1374_1285_735.png (Etiqueta real: 0_ACK)

 Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!

 Subiendo 5 imágenes al bucket 'z72pepee/tfm-explicabilidad'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...IENTS/PAT_1017_97_577.png: 100%|##########| 63.2kB / 63.2kB            

  ...ENTS/PAT_1172_622_420.png: 100%|##########| 67.1kB / 67.1kB            

  ...ENTS/PAT_1259_892_793.png: 100%|##########| 72.8kB / 72.8kB            

  ...NTS/PAT_1300_1056_532.png: 100%|##########| 58.9kB / 58.9kB            

  ...NTS/PAT_1374_1285_735.png: 100%|##########| 70.3kB / 70.3kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_EXPECTED_GRADIENTS/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_EXPECTED_GRADIENTS/", ignore_errors=True)

In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT ...")
explicador = IntegratedGradientsExplainer(
    model=modelo,
    processor=procesador
)




carpeta_salida = "./Resultados_INTEGRATED_GRADIENTS/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando INTEGRATED GRADIENTS para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil,baselines=None, n_steps=50)

        # Guardar en local
        imagen_final = resultado_xai['ig_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_INTEGRATED_GRADIENTS/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT ...

 Procesando Split: test
[1/115] Generando INTEGRATED GRADIENTS para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)
[2/115] Generando INTEGRATED GRADIENTS para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)
[3/115] Generando INTEGRATED GRADIENTS para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)
[4/115] Generando INTEGRATED GRADIENTS para: PAT_1300_1056_532.png (Etiqueta real: 0_ACK)
[5/115] Generando INTEGRATED GRADIENTS para: PAT_1374_1285_735.png (Etiqueta real: 0_ACK)

 Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!

 Subiendo 5 imágenes al bucket 'z72pepee/tfm-explicabilidad'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...IENTS/PAT_1017_97_577.png: 100%|##########| 69.9kB / 69.9kB            

  ...ENTS/PAT_1172_622_420.png: 100%|##########| 72.6kB / 72.6kB            

  ...NTS/PAT_1374_1285_735.png: 100%|##########| 75.0kB / 75.0kB            

  ...NTS/PAT_1300_1056_532.png: 100%|##########| 67.3kB / 67.3kB            

  ...ENTS/PAT_1259_892_793.png: 100%|##########| 72.1kB / 72.1kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_INTEGRATED_GRADIENTS/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_INTEGRATED_GRADIENTS/", ignore_errors=True)

In [ ]:
# Dependencia: grad-cam (instalar con pip)

In [ ]:


# ---  INICIALIZAR LA LIBRERÍA ---
print(" Initializing GlassBox-ViT ...")
explicador = ScoreCamExplainer(
    model=modelo,
    processor=procesador
)




carpeta_salida = "./Resultados_SCORECAM/"
os.makedirs(carpeta_salida, exist_ok=True)

splits_a_procesar = ["test"]
archivos_a_subir = [] # <-- LISTA PARA GUARDAR LAS RUTAS A SUBIR

for split in splits_a_procesar:
    print(f"\n Procesando Split: {split}")

    # Extraemos los nombres de las clases reales
    class_names = data[split].features["label"].names

    # CRUCIAL: Evitamos que HF decodifique la imagen automáticamente
    data[split] = data[split].cast_column("image", HFImage(decode=False))

    # ---  BUCLE DE PROCESAMIENTO ---
    for idx, example in enumerate(data[split]):
        img_bytes = example["image"]["bytes"]
        imagen_pil = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        label_real_id = example["label"]
        label_real_name = class_names[label_real_id]
        ruta_completa = example["image"]["path"]
        nombre_original = os.path.basename(ruta_completa)

        print(f"[{idx+1}/{len(data[split])}] Generando SCORECAM para: {nombre_original} (Etiqueta real: {label_real_name})")

        # Generar LIME
        resultado_xai = explicador.generate(pil_image=imagen_pil)

        # Guardar en local
        imagen_final = resultado_xai['scorecam_image']
        ruta_guardado_local = os.path.join(carpeta_salida, f"{nombre_original}")
        imagen_final.save(ruta_guardado_local)

        # --- PREPARAR PARA SUBIR A HUGGING FACE ---
        # Definimos la ruta donde se guardará DENTRO del bucket:
        # Ej: "dataset1_microsoft/Resultados_LIME/imagen.jpg"
        ruta_en_bucket = f"{nombre_modelo}/Resultados_SCORECAM/{nombre_original}"

        # Añadimos la tupla (ruta_local, ruta_remota) a nuestra lista
        archivos_a_subir.append((ruta_guardado_local, ruta_en_bucket))

        # if idx == 4:
        #     print("\n Parada de seguridad activada (idx == 4). ¡Elimina el 'break' para procesar todo el dataset!")
        #     break

# ---  SUBIDA EN BLOQUE AL BUCKET ---
if archivos_a_subir:
    print(f"\n Subiendo {len(archivos_a_subir)} imágenes al bucket '{bucket_id}'...")
    batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print(" ¡Subida al bucket completada con éxito!")
else:
    print("\n No hay archivos para subir.")

print(f"\n Proceso finalizado. Las imágenes están en local en: {carpeta_salida} y subidas a tu bucket.")

 Initializing GlassBox-ViT ...

 Procesando Split: test
[1/115] Generando SCORECAM para: PAT_1017_97_577.png (Etiqueta real: 0_ACK)


100%|██████████| 48/48 [00:12<00:00,  3.82it/s]


[2/115] Generando SCORECAM para: PAT_1172_622_420.png (Etiqueta real: 0_ACK)


100%|██████████| 48/48 [00:12<00:00,  3.89it/s]


[3/115] Generando SCORECAM para: PAT_1259_892_793.png (Etiqueta real: 0_ACK)


100%|██████████| 48/48 [00:11<00:00,  4.07it/s]


... (9 outputs left out)

  ...ECAM/PAT_1259_892_793.png: 100%|##########| 65.4kB / 65.4kB            

  ...ECAM/PAT_1172_622_420.png: 100%|##########| 78.3kB / 78.3kB            

  ...RECAM/PAT_1017_97_577.png: 100%|##########| 63.2kB / 63.2kB            

 ¡Subida al bucket completada con éxito!

 Proceso finalizado. Las imágenes están en local en: ./Resultados_SCORECAM/ y subidas a tu bucket.


In [ ]:
shutil.rmtree("./Resultados_SCORECAM/", ignore_errors=True)

In [ ]:
shutil.rmtree("./Modelos_Entrenados/", ignore_errors=True)